# Physical Feature Ablation + Paired Bootstrap on Colab (issue #13)

Runs the physical-feature ablation study for Model B: retrains 3 XGBoost variants (Embedding-only 8,960d, Physical-only 4d, Full 8,964d) on the same seed=42 dev split, hyperparameters, and protocol as production Model B, evaluates each on the 514-sample Testset (Spearman/Pearson/MAE/MSE, no rescaling), and paired-bootstraps Full against each ablated variant (1,000 resamples, 95% CI).

This runs on Colab rather than locally because building NT embeddings for the full ~5,145-sequence dev split (train+val+test) needs more RAM/compute than a local machine may have, and benefits from Colab's GPU the same way the ISM sweep (issue #11, `colab_ism_sweep.ipynb`) did.

**One-time setup**: upload these files/folders to a folder in your Google Drive: `NT_sacas9_fintuned_model/` (the whole checkpoint directory), `hybrid_xgb_model.json`, `test_metadata.csv`, `SaCas9_v4.xlsx` (the full 5,145-row dataset - needed to reconstruct the seed=42 dev split; Model A isn't used in this notebook). Set `DRIVE_DIR` below to that folder's path.

**Output**: `ablation_results.json` (keys `ablation_results` and `paired_bootstrap_95ci`) - downloaded automatically at the end and also kept on Drive as a backup. Copy it back into your local `xai_test/` dir.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded model assets

Edit `DRIVE_DIR` to the Drive folder holding `NT_sacas9_fintuned_model/`, `hybrid_xgb_model.json`, `test_metadata.csv`, `SaCas9_v4.xlsx`.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "NT_sacas9_fintuned_model",
    "hybrid_xgb_model.json",
    "test_metadata.csv",
    "SaCas9_v4.xlsx",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

If `apt-get install vienna-rna` fails (package not found on this Colab image), say so and I'll add a conda-forge/bioconda fallback - don't silently skip this step, `Model_B_Predictor` will fail without both binaries on `PATH`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q transformers biopython xgboost openpyxl

## 4. Write the predictor + ablation modules

Mirrors the current committed `xai_test/` repo exactly (embedded at notebook-build time, not fetched at runtime).

In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self.extract_nt_embeddings(sequences)
        physical_feats = self.compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4

# Neither Model_A_Predictor nor Model_B_Predictor batches internally - predict()
# runs whatever list it's given as one forward pass. An ISM sweep can hand it
# tens of thousands of mutants at once, which is large enough to exhaust GPU
# memory (observed: CUDA OOM inside Model A's LSTM forward pass on a 14.56GB
# T4 trying to allocate 20.71 GiB for one such call). Chunking dispatch here
# keeps peak memory bounded regardless of sweep size, without changing either
# predictor's own predict() contract.
DEFAULT_PREDICT_BATCH_SIZE = 256


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def dispatch_in_batches(
    fn, items: list[str], batch_size: int, stack=np.concatenate
) -> np.ndarray:
    """Call `fn()` in bounded chunks and combine the results with `stack`.

    Generic version of the chunking `predict_in_batches` needs for 1D score
    arrays; `step4_ablation.py` reuses this directly (with `stack=np.vstack`)
    to batch `Model_B_Predictor.extract_nt_embeddings()`'s 2D output the same
    way, instead of re-deriving the chunk loop.
    """
    if not items:
        return np.array([])
    chunks = [fn(items[i : i + batch_size]) for i in range(0, len(items), batch_size)]
    return stack(chunks)


def predict_in_batches(predictor, sequences: list[str], batch_size: int) -> np.ndarray:
    """Dispatch `predictor.predict()` in bounded chunks (shared with mismatch_profiling.py)."""
    if not sequences:
        return np.array([], dtype=np.float64)

    def _predict_chunk(chunk):
        return np.asarray(predictor.predict(chunk), dtype=np.float64)

    return dispatch_in_batches(
        _predict_chunk, sequences, batch_size, stack=np.concatenate
    )


def relative_delta(
    wt_scores: np.ndarray, mutant_scores: np.ndarray, nan_threshold: float
) -> np.ndarray:
    """`(mutant - wt) / |wt|` per sample, NaN-masked wherever `|wt| <= nan_threshold`.

    `wt_scores` is 1D (one value per sample); `mutant_scores` broadcasts against
    it along its leading axis - shape `(n, positions, alt_bases)` for the ISM
    sweep, `(n, scenarios)` for complex mismatch profiling. Shared so both
    modules apply the exact same guardrail (see `RELATIVE_DELTA_NAN_THRESHOLD`).
    """
    broadcast_shape = (wt_scores.shape[0],) + (1,) * (mutant_scores.ndim - 1)
    wt_broadcast = wt_scores.reshape(broadcast_shape)
    with np.errstate(divide="ignore", invalid="ignore"):
        delta = (mutant_scores - wt_broadcast) / np.abs(wt_broadcast)
    delta[np.abs(wt_scores) <= nan_threshold] = np.nan
    return delta


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`. `predict_batch_size` bounds how many sequences reach
    `predictor.predict()` per call; lower it if you still hit an
    out-of-memory error.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_ism_sweep(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(
            sequences, predictor_a, predict_batch_size=predict_batch_size
        ),
        "ism_delta_model_b": compute_ism_delta(
            sequences, predictor_b, predict_batch_size=predict_batch_size
        ),
    }


In [ ]:
%%writefile test_dataset_verifier.py
import hashlib
import os
import sys

import pandas as pd
import torch


def generate_sha256(sequence: str) -> str:
    return hashlib.sha256(sequence.encode('utf-8')).hexdigest()

def load_dataset(file_path: str) -> pd.DataFrame:
    if file_path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(file_path)
    return pd.read_csv(file_path)

def standardize_columns(full_df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names (`sequence`/`true_score`) and preserve `original_id`."""
    rename_mapping = {}
    if 'sequence' not in full_df.columns and 'input_sequence' in full_df.columns:
        rename_mapping['input_sequence'] = 'sequence'
    if 'true_score' not in full_df.columns and 'score' in full_df.columns:
        rename_mapping['score'] = 'true_score'

    if rename_mapping:
        full_df = full_df.rename(columns=rename_mapping)

    if 'original_id' not in full_df.columns:
        full_df['original_id'] = full_df.index

    return full_df

def compute_split_indices(dataset_size: int, seed: int = 42) -> tuple[list[int], list[int]]:
    """The seed=42 `torch.randperm` split isolating the Testset from the dev pool.

    Returns `(train_val_indices, test_indices)`: `test_indices` (the last 10%
    of the permutation) is exactly what `test_metadata.csv` is derived from;
    `train_val_indices` (the first 90%) is the disjoint dev pool other issues
    (e.g. #13's ablation retraining) build train/val splits from - reusing
    this function guarantees they stay leak-free against the Testset.
    """
    test_size = max(1, int(dataset_size * 0.1))
    train_val_size = dataset_size - test_size

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(dataset_size, generator=generator).tolist()
    return indices[:train_val_size], indices[train_val_size:]

def isolate_and_verify_testset(full_dataset_path: str, output_csv_path: str = 'test_metadata.csv', seed: int = 42):
    print(f"Loading full dataset from {full_dataset_path}...")
    full_df = load_dataset(full_dataset_path)
    dataset_size = len(full_df)

    # 1. 컬럼 매핑 표준화 및 원본 행 인덱스 보존
    full_df = standardize_columns(full_df)

    # 2. PyTorch random_split 재현 (int(dataset_size * 0.1) = 514개)
    _, test_indices = compute_split_indices(dataset_size, seed)
    test_size = len(test_indices)

    # 3. 테스트셋 추출 및 크기 검증
    test_df = full_df.iloc[test_indices].copy().reset_index(drop=True)
    if len(test_df) != test_size:
        raise ValueError(f"Split error: Expected {test_size} test sequences, but got {len(test_df)}.")

    # 4. sample_id (0 ~ test_size-1) 부여 및 서열 해시 생성
    test_df['sample_id'] = range(len(test_df))
    test_df['sequence_hash'] = test_df['sequence'].apply(generate_sha256)

    # 5. 전체 데이터셋 단위 SHA256 체크섬
    dataset_hash = hashlib.sha256(pd.util.hash_pandas_object(test_df['sequence']).values).hexdigest()
    print(f"Testset isolation completed: {len(test_df)} samples")
    print(f"Dataset SHA256 Checksum: {dataset_hash}")

    # 6. 필수 메타데이터 칼럼 저장
    required_cols = ['sample_id', 'original_id', 'sequence', 'true_score', 'sequence_hash']
    for col in required_cols:
        if col not in test_df.columns:
            raise KeyError(f"Missing required column in dataset: {col}")

    meta_df = test_df[required_cols]
    meta_df.to_csv(output_csv_path, index=False)
    print(f"Saved isolated test metadata to {output_csv_path}")

    return meta_df, dataset_hash

if __name__ == "__main__":
    script_dir = os.path.dirname(os.path.abspath(__file__))

    if len(sys.argv) > 1:
        target_file = sys.argv[1]
        if not os.path.isabs(target_file):
            target_file = os.path.join(script_dir, target_file)
    else:
        target_file = os.path.join(script_dir, "SaCas9_v4.xlsx")

    try:
        isolate_and_verify_testset(target_file)
    except FileNotFoundError:
        print(f"File not found: {target_file}")
        print(f"현재 스크립트 위치({script_dir})에 데이터 파일이 존재하는지 확인해 주세요.")

In [ ]:
%%writefile dev_split.py
"""Dev split (train/val) for Model B ablation retraining (Step 4-1, issue #13).

The train.csv/val.csv/test.csv checked into this project predate the current
seed=42 Testset isolation (test_dataset_verifier.py / test_metadata.csv) and
don't correspond to it: 460 of the Testset's 464 unique sequences also
appear in train.csv. Training the ablation variants on that split would mean
they'd already seen ~99% of the Testset's unique content, invalidating the
Testset evaluation this issue's paired bootstrap depends on.

Instead, `build_dev_split` reconstructs train/val fresh from the same
seed=42 permutation `test_dataset_verifier.compute_split_indices` uses to
isolate the Testset. `train_val_indices` and `test_indices` are complementary
halves of one `torch.randperm` call, so the dev split is disjoint from the
Testset by construction - no post-hoc filtering needed. Val is sized to
match the Testset (514 rows); the remainder is train.
"""

import pandas as pd
from test_dataset_verifier import (
    compute_split_indices,
    load_dataset,
    standardize_columns,
)


def build_dev_split(
    full_dataset_path: str, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns `(train_df, val_df)`, both disjoint from the seed=42-isolated Testset."""
    full_df = standardize_columns(load_dataset(full_dataset_path))
    dataset_size = len(full_df)

    train_val_indices, test_indices = compute_split_indices(dataset_size, seed)
    val_size = len(test_indices)
    train_indices = train_val_indices[:-val_size]
    val_indices = train_val_indices[-val_size:]

    train_df = full_df.iloc[train_indices].reset_index(drop=True)
    val_df = full_df.iloc[val_indices].reset_index(drop=True)
    return train_df, val_df


In [ ]:
%%writefile paired_bootstrap.py
"""Paired bootstrap significance testing for Model B ablation (Step 4-1, issue #13).

Each bootstrap draw resamples the SAME index set (with replacement) for
y_true and every compared variant's predictions together, so the comparison
stays paired - it isolates the metric delta itself rather than treating the
two variants as independent samples with their own sampling noise.
"""

from collections.abc import Callable

import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

MetricFn = Callable[[np.ndarray, np.ndarray], float]

METRIC_FUNCS: dict[str, MetricFn] = {
    "spearman": lambda y_true, y_pred: spearmanr(y_true, y_pred)[0],
    "pearson": lambda y_true, y_pred: pearsonr(y_true, y_pred)[0],
    "mae": mean_absolute_error,
    "mse": mean_squared_error,
}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Spearman/Pearson/MAE/MSE on raw values - no cross-model rescaling."""
    return {name: float(fn(y_true, y_pred)) for name, fn in METRIC_FUNCS.items()}


def paired_bootstrap_delta_ci(
    y_true: np.ndarray,
    preds_a: np.ndarray,
    preds_b: np.ndarray,
    metric: str,
    n_resamples: int = 1000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple[float, float]:
    """CI for `metric(preds_a) - metric(preds_b)` under paired resampling.

    `metric` must be a key of `METRIC_FUNCS`. Returns `(lower, upper)` for the
    `ci` central interval (default 95%) over `n_resamples` bootstrap draws.
    """
    if metric not in METRIC_FUNCS:
        raise ValueError(
            f"Unknown metric: {metric!r}. Choose from {sorted(METRIC_FUNCS)}."
        )

    metric_fn = METRIC_FUNCS[metric]
    y_true = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)
    n = len(y_true)

    rng = np.random.default_rng(seed)
    deltas = np.empty(n_resamples)
    for i in range(n_resamples):
        idx = rng.integers(0, n, size=n)
        deltas[i] = metric_fn(y_true[idx], preds_a[idx]) - metric_fn(
            y_true[idx], preds_b[idx]
        )

    alpha = 1 - ci
    lower, upper = np.quantile(deltas, [alpha / 2, 1 - alpha / 2])
    return float(lower), float(upper)


In [ ]:
%%writefile ablation.py
"""Model B physical-feature ablation + paired bootstrap (Step 4-1, issue #13).

Trains three XGBoost regressors - Embedding-only (8,960d), Physical-only
(4d), and Full (8,964d) - on the same dev split, seed, and hyperparameters
as production Model B (`nt_feature_stacking.py`'s Phase 4), then paired-
bootstraps Full against each ablated variant on the Testset. No cross-model
or cross-variant score rescaling is applied anywhere in this module.
"""

import numpy as np
import xgboost as xgb
from paired_bootstrap import compute_metrics, paired_bootstrap_delta_ci

# Same protocol as nt_feature_stacking.py's Phase 4 XGBoost fit - shared here
# so all three ablation variants (and production Model B) are provably using
# identical hyperparameters, not three independently-typed copies of them.
XGB_HYPERPARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "tree_method": "hist",
    "early_stopping_rounds": 20,
}

EMBEDDING_DIM = 8960
PHYSICAL_DIM = 4

# Bootstrap deltas are always Full minus the named ablated variant, so a
# negative delta_mse / positive delta_spearman means Full is better.
BOOTSTRAP_METRICS = ("spearman", "mse")


def split_ablation_features(
    X_full: np.ndarray, embedding_dim: int = EMBEDDING_DIM
) -> dict[str, np.ndarray]:
    """Slice a `(n, embedding_dim + physical_dim)` feature matrix into the 3 variants' inputs."""
    return {
        "embedding": X_full[:, :embedding_dim],
        "physical": X_full[:, embedding_dim:],
        "full": X_full,
    }


def train_ablation_variant(
    X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray
) -> xgb.XGBRegressor:
    """Train one XGBoost regressor under the shared `XGB_HYPERPARAMS` protocol."""
    model = xgb.XGBRegressor(**XGB_HYPERPARAMS)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model


def run_ablation(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    bootstrap_seed: int = 42,
    embedding_dim: int = EMBEDDING_DIM,
) -> dict:
    """Train all 3 variants, evaluate on the Testset, and paired-bootstrap Full vs each.

    Returns a dict with `ablation_results` (per-variant Spearman/Pearson/MAE/MSE
    on the Testset, raw/unrescaled) and `paired_bootstrap_95ci` (95% CI for
    `metric(full) - metric(variant)`, for each of `BOOTSTRAP_METRICS`).
    `embedding_dim` only needs overriding in tests against synthetic,
    lower-dimensional feature matrices.
    """
    train_splits = split_ablation_features(X_train, embedding_dim)
    val_splits = split_ablation_features(X_val, embedding_dim)
    test_splits = split_ablation_features(X_test, embedding_dim)

    # These three key names ("embedding_only"/"physical_only"/"full_model") are
    # mandated verbatim by the Standard Export Schema (plan_realize.md section
    # 4, `ablation_results`), which issue #17 must reproduce exactly. This is
    # a deliberate exception to CONTEXT.md's glossary guidance to avoid
    # "embedding_only" (there, it warns against conflating it with "NT-only
    # regression head performance", a different, unrelated pipeline) - kept
    # here only because it's always nested under `ablation_results`, which
    # disambiguates it as the Step 4(1) ablation variant, never that other
    # metric.
    result_key = {
        "embedding": "embedding_only",
        "physical": "physical_only",
        "full": "full_model",
    }

    metrics_by_variant = {}
    predictions_by_variant = {}
    for name in ("embedding", "physical", "full"):
        model = train_ablation_variant(
            train_splits[name], y_train, val_splits[name], y_val
        )
        preds = model.predict(test_splits[name])
        predictions_by_variant[name] = preds
        metrics_by_variant[name] = compute_metrics(y_test, preds)

    bootstrap_cis = {}
    for name in ("embedding", "physical"):
        for metric in BOOTSTRAP_METRICS:
            lower, upper = paired_bootstrap_delta_ci(
                y_test,
                predictions_by_variant["full"],
                predictions_by_variant[name],
                metric=metric,
                seed=bootstrap_seed,
            )
            bootstrap_cis[f"delta_{metric}_full_vs_{name}_only"] = [lower, upper]

    return {
        "ablation_results": {
            result_key[name]: metrics_by_variant[name]
            for name in ("embedding", "physical", "full")
        },
        "paired_bootstrap_95ci": bootstrap_cis,
    }


## 5. Confirm GPU is available

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - Runtime > Change runtime type > GPU")

## 6. Build the dev split (train/val) and load the Testset

`build_dev_split` reconstructs train/val from the same seed=42 permutation that isolates the Testset (`test_metadata.csv`), so the dev split is disjoint from it by construction - unrelated to (and safer than) the repo's checked-in `train.csv`/`val.csv`, which predate this Testset isolation and were found to leak into it.

In [ ]:
import pandas as pd
from dev_split import build_dev_split

train_df, val_df = build_dev_split("SaCas9_v4.xlsx")
test_df = pd.read_csv("test_metadata.csv")

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 7. Extract features (NT embeddings + physical) for train/val/test

`extract_nt_embeddings()` runs its whole input list through the NT transformer in one forward pass - unbounded memory, same shape as the OOM issue #11 hit in `Model_A_Predictor.predict()`. `dispatch_in_batches` (from `ism_sweep.py`) bounds this the same way; lower `PREDICT_BATCH_SIZE` further if you still hit a CUDA OOM. `compute_physical_features()` is already internally batched (RNAfold/RNAduplex over stdin) and runs on CPU regardless.

In [ ]:
from model_b_wrapper import Model_B_Predictor

predictor = Model_B_Predictor(
    nt_model_dir="./NT_sacas9_fintuned_model", xgb_model_path="hybrid_xgb_model.json"
)

In [ ]:
import time

import numpy as np
from ism_sweep import dispatch_in_batches

PREDICT_BATCH_SIZE = 128  # lower (e.g. 32) if you still hit CUDA OOM


def build_features(sequences):
    embeddings = dispatch_in_batches(
        predictor.extract_nt_embeddings, sequences, PREDICT_BATCH_SIZE, stack=np.vstack
    )
    physical = predictor.compute_physical_features(sequences)
    return np.hstack([embeddings, physical])


t0 = time.time()
X_train = build_features(train_df["sequence"].tolist())
X_val = build_features(val_df["sequence"].tolist())
X_test = build_features(test_df["sequence"].tolist())
y_train = train_df["true_score"].to_numpy()
y_val = val_df["true_score"].to_numpy()
y_test = test_df["true_score"].to_numpy()
elapsed = time.time() - t0
print(f"Feature extraction completed in {elapsed / 60:.1f} min")
print(f"X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

## 8. Run the ablation + paired bootstrap

In [ ]:
from ablation import run_ablation

t0 = time.time()
result = run_ablation(X_train, y_train, X_val, y_val, X_test, y_test)
elapsed = time.time() - t0
print(f"Ablation completed in {elapsed / 60:.1f} min")

print("\n=== Ablation Results (Testset) ===")
for variant, metrics in result["ablation_results"].items():
    print(f" - {variant}: {metrics}")
print("\n=== Paired Bootstrap 95% CI (Full vs each variant) ===")
for key, ci in result["paired_bootstrap_95ci"].items():
    print(f" - {key}: [{ci[0]:.4f}, {ci[1]:.4f}]")

## 9. Save and download the result

In [ ]:
import json

with open("ablation_results.json", "w") as f:
    json.dump(result, f, indent=2)
# Also keep a copy on Drive in case the runtime disconnects before you download.
with open(os.path.join(DRIVE_DIR, "ablation_results.json"), "w") as f:
    json.dump(result, f, indent=2)

from google.colab import files

files.download("ablation_results.json")

Copy the downloaded `ablation_results.json` into your local `xai_test/` directory - it has the same schema `step4_ablation.py` produces locally.